Visualización sobre la evolución de los ratings en kdramas según su año y plataforma,

Gráfico de líneas múltiples
Eje X → año
Eje Y → rating promedio
Color → plataforma

In [21]:
import pandas as pd
import altair as alt
df = pd.read_csv("2016-2025_Kdramas_dataset.csv")
df = df.dropna(subset=["release_year", "ratings"])

df["release_year"] = df["release_year"].astype(int)

df["ratings"] = df["ratings"].astype(float)
platforms = [
    "netflix",
    "viu",
    "prime_video",
    "disney_plus"
]
df_long = df.melt(
    id_vars=["release_year", "ratings"],
    value_vars=platforms,
    var_name="platform",
    value_name="available"
)
df_long = df_long[df_long["available"] == "yes"] # Corrected 'Yes' to 'yes'
ratings_year = df_long.groupby(
    ["release_year", "platform"]
).agg(
    avg_rating=('ratings', 'mean'),
    kdrama_count=('ratings', 'count')
).reset_index()

# Define a selection for interactive highlighting on hover
highlight = alt.selection_point(on='mouseover', fields=['platform'], nearest=True)

# Define the common encoding for x, y, color, and tooltip
base_encoding = alt.Chart(ratings_year).encode(
    x=alt.X(
        'release_year:O',
        title='Año'
    ),
    y=alt.Y(
        'avg_rating:Q',
        title='Rating promedio'
    ),
    color=alt.Color(
        'platform:N',
        title='Plataforma'
    ),
    tooltip=[
        alt.Tooltip('release_year:O', title='Año'),
        alt.Tooltip('platform:N', title='Plataforma'),
        alt.Tooltip('avg_rating:Q', title='Rating promedio', format='.2f'),
        alt.Tooltip('kdrama_count:Q', title='Cantidad de K-dramas')
    ]
).properties(
    width=750,
    height=450,
    title='Evolución del rating promedio de K-dramas según plataforma (2016-2025)'
).interactive() # Keep interactive for pan and zoom

# Create the line layer with conditional styling based on selection
lines = base_encoding.mark_line().encode(
    strokeWidth=alt.condition(highlight, alt.value(4), alt.value(2)), # Thicker line on hover
    opacity=alt.condition(highlight, alt.value(1), alt.value(0.2)) # More opaque on hover
)

# Create the point layer for better visibility and interaction
points = base_encoding.mark_point(filled=True, size=80).encode(
    opacity=alt.condition(highlight, alt.value(1), alt.value(0)), # Points become visible/opaque on hover
    color=alt.condition(highlight, 'platform:N', alt.value('lightgray')) # Color if selected, else gray
)

# Combine the lines and points layers and add the selection parameter
chart = (lines + points).add_params(highlight)

chart

!pip install vl-convert-python

chart.save("visualizacion.html")

chart.save("visualizacion.png")

In [23]:
from PIL import Image

# Open the PNG image
png_image = Image.open("visualizacion.png")

# Convert to RGB (JPG does not support transparency)
rgb_image = png_image.convert("RGB")

# Save as JPG
rgb_image.save("visualizacion.jpg")

print("Gráfico guardado como visualizacion.jpg")

from google.colab import files

ratings_year.to_csv("ratings_kdramas.csv", index=False)

files.download("ratings_kdramas.csv")



Gráfico guardado como visualizacion.jpg


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>